In [1]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.transforms import RandomErasing
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score
)
from tqdm import tqdm
import warnings
import random
import re
from collections import defaultdict, Counter
import time

warnings.filterwarnings('ignore')

# -------------------- Deterministic Seed --------------------
def set_deterministic(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# -------------------- Datasets --------------------
class OriginalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None, target_size=(224,224)):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.target_size = target_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        if img is None:
            img = np.zeros((*self.target_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_AREA)
        img = Image.fromarray(img)
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# -------------------- Model Factory --------------------
def create_model(model_name, num_classes=2, pretrained=True):
    """
    Returns a model with modified classifier head for binary classification.
    Supported: 'resnet18', 'resnet101', 'mobilenet_v2', 'efficientnet_b2', 'vgg19'
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if model_name == 'resnet18':
        model = models.resnet18(pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
    elif model_name == 'resnet101':
        model = models.resnet101(pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
    elif model_name == 'mobilenet_v2':
        model = models.mobilenet_v2(pretrained=pretrained)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
    elif model_name == 'efficientnet_b2':
        model = models.efficientnet_b2(pretrained=pretrained)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
    elif model_name == 'vgg19':
        model = models.vgg19(pretrained=pretrained)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    model = model.to(device)
    return model

# -------------------- Training Function --------------------
def train_model(model, train_loader, test_loader, model_name="Model", epochs=15):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nTraining {model_name} on {device}...")
    model = model.to(device)

    # Allow fine-tuning of all layers
    for param in model.parameters():
        param.requires_grad = True

    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_acc = 0.0
    best_model_state = None

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        for inputs, targets in tqdm(train_loader, desc=f"{model_name} Epoch {epoch}/{epochs}"):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        # Validation
        model.eval()
        all_preds, all_targets = [], []
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())

        acc = accuracy_score(all_targets, all_preds)
        avg_train_loss = running_loss / len(train_loader.dataset)
        avg_val_loss = val_loss / len(test_loader.dataset)
        print(f"Epoch {epoch}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {acc*100:.2f}%")
        scheduler.step(avg_val_loss)

        if acc > best_acc:
            best_acc = acc
            best_model_state = model.state_dict().copy()

    model.load_state_dict(best_model_state)
    return model, best_acc

# -------------------- Evaluation Function --------------------
def evaluate_model(model, test_loader, device):
    """
    Computes Accuracy, Precision, Recall, F1, AUC,
    model parameters, model size, and inference speed.
    """
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []

    # Measure inference speed
    start_time = time.perf_counter()
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    end_time = time.perf_counter()
    total_time = end_time - start_time
    num_images = len(test_loader.dataset)
    avg_time_per_image = total_time / num_images

    accuracy = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average='binary', pos_label=1)
    recall = recall_score(all_targets, all_preds, average='binary', pos_label=1)
    f1 = f1_score(all_targets, all_preds, average='binary', pos_label=1)
    auc = roc_auc_score(all_targets, all_probs)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    model_size_mb = (total_params * 4) / (1024 * 1024)   # float32

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'total_params': total_params,
        'trainable_params': trainable_params,
        'model_size_mb': model_size_mb,
        'inference_speed_sec_per_img': avg_time_per_image,
        'predictions': all_preds,
        'targets': all_targets,
        'probabilities': all_probs,
    }
    return metrics

# ==========================================================
# MAIN EXECUTION
# ==========================================================
set_deterministic(42)

base_path = '/kaggle/input/datasets/iarunava/cell-images-for-detecting-malaria/cell_images'
categories = ['Parasitized', 'Uninfected']

# Build full list of paths and labels
image_paths, labels = [], []
for cat_idx, cat in enumerate(categories):
    folder = os.path.join(base_path, cat)
    exts = ('.png', '.jpg', '.jpeg')
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(exts)]
    image_paths.extend(files)
    labels.extend([cat_idx] * len(files))

# -------------------- PATIENT-AWARE SPLIT --------------------
path_to_label = {path: label for path, label in zip(image_paths, labels)}
patient_to_images = defaultdict(list)

for path in image_paths:
    match = re.search(r'P(\d+)', os.path.basename(path))
    if match:
        patient_id = f"P{match.group(1)}"
    else:
        patient_id = os.path.basename(path).split('_')[0]
    patient_to_images[patient_id].append(path)

unique_patients = list(patient_to_images.keys())

patient_majority_label = {}
for pid, paths in patient_to_images.items():
    labels_for_patient = [path_to_label[p] for p in paths]
    majority = max(set(labels_for_patient), key=labels_for_patient.count)
    patient_majority_label[pid] = majority

patient_labels = [patient_majority_label[pid] for pid in unique_patients]

print(f"Total unique patients: {len(unique_patients)}")

train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=0.20,
    random_state=42,
    stratify=patient_labels
)

train_paths, train_labels = [], []
test_paths, test_labels = [], []

for pid in train_patients:
    for path in patient_to_images[pid]:
        train_paths.append(path)
        train_labels.append(path_to_label[path])

for pid in test_patients:
    for path in patient_to_images[pid]:
        test_paths.append(path)
        test_labels.append(path_to_label[path])

print("\n" + "="*70)
print("DEBUGGING DATASET SPLIT")
print("="*70)
print(f"Total images        : {len(image_paths)}")
print(f"Training images     : {len(train_paths)}")
print(f"Testing images      : {len(test_paths)}")
print("\nOverall class distribution:", Counter(labels))
print("Training class distribution:", Counter(train_labels))
print("Testing class distribution :", Counter(test_labels))
print("="*70)

# -------------------- Transforms --------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# -------------------- Datasets & DataLoaders --------------------
train_orig = OriginalDataset(train_paths, train_labels, transform=train_transform)
test_orig  = OriginalDataset(test_paths, test_labels, transform=test_transform)

batch_size = 64
train_loader = DataLoader(train_orig, batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_orig,  batch_size, shuffle=False, num_workers=4, pin_memory=True)

# -------------------- Benchmarking Multiple Models --------------------
models_to_train = [
    'resnet18',
    'mobilenet_v2',
    'efficientnet_b2',
    'resnet101',
    'vgg19'
]

epochs = 15  # You can increase for better performance

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = {}

# <-- ADDED: track best model
best_acc = 0.0
best_model_state = None
best_model_name = None

for model_name in models_to_train:
    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()}")
    print('='*60)

    model = create_model(model_name, num_classes=2, pretrained=True)
    model, best_acc_internal = train_model(model, train_loader, test_loader,
                                  model_name=model_name.upper(), epochs=epochs)

    # Evaluate
    metrics = evaluate_model(model, test_loader, device)
    results[model_name] = metrics

    # Print per-model summary
    print(f"\n--- {model_name.upper()} Evaluation ---")
    print(f"Accuracy  : {metrics['accuracy']*100:.2f}%")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"F1        : {metrics['f1']:.4f}")
    print(f"AUC       : {metrics['auc']:.4f}")
    print(f"Params    : {metrics['total_params']:,}")
    print(f"Model size: {metrics['model_size_mb']:.2f} MB")
    print(f"Inference : {metrics['inference_speed_sec_per_img']*1000:.2f} ms/img")

    # <-- ADDED: update best model if current is better
    if metrics['accuracy'] > best_acc:
        best_acc = metrics['accuracy']
        best_model_state = model.state_dict().copy()
        best_model_name = model_name

# -------------------- Final Comparison Table --------------------
print("\n" + "="*80)
print("FINAL BENCHMARKING SUMMARY (Patient-Aware Split)")
print("="*80)

print(f"{'Model':<18} {'Acc%':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6} {'Params (M)':>10} {'Size (MB)':>9} {'Speed (ms)':>10}")
print("-"*80)

for model_name, m in results.items():
    print(f"{model_name:<18} {m['accuracy']*100:>6.2f} {m['precision']:>6.4f} {m['recall']:>6.4f} {m['f1']:>6.4f} {m['auc']:>6.4f} {m['total_params']/1e6:>10.2f} {m['model_size_mb']:>9.2f} {m['inference_speed_sec_per_img']*1000:>10.2f}")

print("="*80)

# Optionally, show a classification report for each model? We'll just show for the best one.
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
print(f"\nBest model by Accuracy: {best_model_name.upper()} ({results[best_model_name]['accuracy']*100:.2f}%)")

# Show classification report for the best model
print("\n" + "="*60)
print(f"CLASSIFICATION REPORT – {best_model_name.upper()}")
print("="*60)
print(classification_report(results[best_model_name]['targets'],
                            results[best_model_name]['predictions'],
                            target_names=['Uninfected', 'Parasitized']))

print("\nConfusion Matrix (Best Model):")
print(confusion_matrix(results[best_model_name]['targets'],
                       results[best_model_name]['predictions']))

# <-- ADDED: Save the best model for explainability
if best_model_state is not None:
    torch.save(best_model_state, 'best_model.pth')
    with open('best_model_name.txt', 'w') as f:
        f.write(best_model_name)
    print(f"\n✅ Best model saved as 'best_model.pth' with name '{best_model_name}' (accuracy = {best_acc*100:.2f}%)")
else:
    print("\n⚠️ No model saved (no models were trained).")

Total unique patients: 200

DEBUGGING DATASET SPLIT
Total images        : 27558
Training images     : 21426
Testing images      : 6132

Overall class distribution: Counter({0: 13779, 1: 13779})
Training class distribution: Counter({1: 11044, 0: 10382})
Testing class distribution : Counter({0: 3397, 1: 2735})

Training RESNET18
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]



Training RESNET18 on cuda...


RESNET18 Epoch 1/15: 100%|██████████| 335/335 [01:25<00:00,  3.91it/s]


Epoch 1: Train Loss: 0.3245 | Val Loss: 0.2870 | Val Acc: 95.68%


...

Epoch 13: Train Loss: 0.2383 | Val Loss: 0.2608 | Val Acc: 96.44%


VGG19 Epoch 14/15: 100%|██████████| 335/335 [06:48<00:00,  1.22s/it]


Epoch 14: Train Loss: 0.2390 | Val Loss: 0.2582 | Val Acc: 96.90%


VGG19 Epoch 15/15: 100%|██████████| 335/335 [06:48<00:00,  1.22s/it]


Epoch 15: Train Loss: 0.2380 | Val Loss: 0.2607 | Val Acc: 96.69%

--- VGG19 Evaluation ---
Accuracy  : 96.69%
Precision : 0.9393
Recall    : 0.9898
F1        : 0.9639
AUC       : 0.9933
Params    : 139,578,434
Model size: 532.45 MB
Inference : 6.65 ms/img

FINAL BENCHMARKING SUMMARY (Patient-Aware Split)
Model                Acc%   Prec    Rec     F1    AUC Params (M) Size (MB) Speed (ms)
--------------------------------------------------------------------------------
resnet18            96.92 0.9492 0.9835 0.9661 0.9932      11.18     42.64       1.42
mobilenet_v2        96.85 0.9507 0.9803 0.9653 0.9900       2.23      8.49       1.35
efficientnet_b2     97.00 0.9525 0.9817 0.9669 0.9911       7.70     29.39       2.13
resnet101           96.75 0.9512 0.9773 0.9641 0.9902      42.50    162.14       5.79
vgg19               96.69 0.9393 0.9898 0.9639 0.9933     139.58    532.45       6.65

Best model by Accuracy: EFFICIENTNET_B2 (97.00%)

CLASSIFICATION REPORT – EFFICIENTNET_B2
     